In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Colab_Notebooks/semester_project
%ls

Mounted at /content/drive
/content/drive/MyDrive/Colab_Notebooks/semester_project
EWZ_Daily_Preprocessed.csv  try_chronos.ipynb


In [3]:
!pip install chronos-forecasting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 136.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 6.8 MB/s eta 0:00:00


In [1]:
import pandas as pd
from chronos import Chronos2Pipeline
import torch
import numpy as np

c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../../data/EWZ_Daily_Preprocessed.csv")


In [32]:
df.rename(columns={"Name": "timestamp"}, inplace=True)

df.head()

,timestamp,Wärmezähler UST 10 Spitalstrasse 6 Regionalspital Surselva WZ11,Wärmezähler Fernleitung WZ6,Wärmezähler UST 01 Bahnhofstrasse 14,Wärmezähler UST 11 Via Schlifras 46,Wärmezähler UST 12 Via Schlifras 48,Wärmezähler UST 13 Via Schlifras 50,Wärmezähler UST 14 Via Schlifras 54,Wärmezähler UST 15 Solaranlage BWW,Wärmezähler UST 16 Via Schlifras 66/68/70,...,Wärmezähler UST 52 Städtlistrasse 3 / Rathausgasse 2,Wärmezähler UST 53 Via Rolf Maibach 4,"Wärmezähler UST 54 Spitalstrasse 2, Haus Theresia/Fidel",Wärmezähler UST 55 Via Schlifras 71,Wärmezähler UST 56 Via Hans Erni 10,Wärmezähler UST 57 Via Hans Erni 4,Wärmezähler UST 58 Via Schlifras 73,Wärmezähler UST 59 Via Schlifras 45,Wärmezähler UST 60 Via Hans Erni 8,Wärmezähler UST 61 Via Schlifras 55
0,2016-04-01,1.6835,15.7920,0.3020,0.084,0.125,0.180,0.166,0.1387,0.2795,...,0.2369,0.096,0.1894,0.0480,0.0578,0.053,0.035,0.020,0.0521,0.0626
1,2016-04-02,1.8121,16.9004,0.3200,0.088,0.150,0.196,0.195,0.1464,0.3320,...,0.2544,0.115,0.2166,0.0660,0.0532,0.062,0.037,0.028,0.0470,0.0684
2,2016-04-03,1.6738,15.9902,0.3000,0.080,0.155,0.159,0.148,0.1312,0.2750,...,0.2252,0.077,0.1785,0.0510,0.0610,0.077,0.034,0.027,0.0413,0.0700
3,2016-04-04,1.7487,15.7100,0.2790,0.079,0.158,0.171,0.185,0.1335,0.2490,...,0.2302,0.096,0.1435,0.0626,0.0524,0.056,0.035,0.019,0.0451,0.0600
4,2016-04-05,1.7051,15.1992,0.2848,0.080,0.130,0.179,0.162,0.1493,0.2600,...,0.2218,0.112,0.1820,0.0484,0.0576,0.052,0.032,0.027,0.0448,0.0630


In [37]:
split_idx = int(len(df) * 0.85) +1

# Split the dataframe
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

train_df.drop(columns='timestamp', inplace=True)
test_df.drop(columns='timestamp', inplace=True)

# targets = train_df.columns[1:]
# test_df.drop(columns=train_df.columns[1:], inplace=True)


In [38]:
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

In [39]:
context = torch.tensor(train_df.values.T, dtype=torch.float32)
N, T = context.shape
context = context.reshape(N, 1, T)

# Generate multivariate forecast
forecast = np.array(pipeline.predict(
    inputs=context,
    prediction_length=len(test_df),
))

In [40]:
forecast.shape

(59, 1, 21, 525)

In [41]:
forecast = np.mean(forecast, axis=2)

forecast.shape

(59, 1, 525)

In [43]:
y_true = test_df.values.T

N, T = y_true.shape
y_true = y_true.reshape(N, 1, T)

y_true.shape

(59, 1, 525)

In [44]:
np.save("./pred_chronos.npy", forecast)
np.save("./true_chronos.npy", y_true)